In [195]:
import itertools

import numpy as np
import torch
from torch import nn

### 2D - spinful
**Note: This is not a formal proof (see OverLeaf for that), just notes for myself to keep in mind when coding.**

Let $\widetilde{\Lambda} = \mathbb{Z}^2 \times \{\uparrow, \downarrow\}$
Then $\Psi: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$ can be written as:
$$\Psi = F_1 g_1 + F_2 g_2,$$
where $F_1 = \Re \{F\}$, $F_2 = \Im \{F\}$, and $F$ is defined by:
$$F(\mathbf{X}) = \prod_{i<j}\left(\widetilde{f}(X_i) - \widetilde{f}(X_j)\right)$$
where $\widetilde{f}: \widetilde{\Lambda} \rightarrow \mathbb{C}$ is injective in the spatial coordinates of the $L\times L$ cell.
Meanwhile, the symmetric functions $g_1, g_2$ are defined by:
$$g_i(\mathbf{X}) = g_i'\left(\sum_{\boldsymbol{s}\in\Lambda} w_{\boldsymbol{s}} \#\{i | X_i = \boldsymbol{s} \, (\bmod L)\, \}\right)$$
where $\Lambda$ is the restriction of the spatial coordinates in $\widetilde{\Lambda}$ to the $L\times L$ cell, i.e. $\Lambda = Z_{L} \times \{\uparrow, \downarrow\}$. Furthermore, because $\Psi$ must be periodic, we require that $\widetilde{f}$ is also periodic. Practically, we can do so by transforming the spatial coordinates into a complex exponential with the appropriate frequency before inputting them into the NN.

In [209]:
# generic MLP helper class
class MLP(nn.Module):
    def __init__(
        self,
        hidden_layers: int,
        input_dim: int,
        dim_feedforward: int,
        output_dim: int,
        activation: nn.Module = nn.GELU,
        device=torch.device("cpu"),
    ):
        super().__init__()
        # make it
        net = [
            nn.Linear(input_dim, dim_feedforward),
            activation(),
        ]
        for _ in range(hidden_layers):
            net.append(nn.Linear(dim_feedforward, dim_feedforward))
            net.append(activation())
        net.append(nn.Linear(dim_feedforward, output_dim))
        self.MLP = nn.Sequential(*net)
        self.to(device)

    # def _initialize_weights(m: nn.Module):
    #     if isinstance(m, nn.Linear):

    def forward(self, x: torch.Tensor):
        return self.MLP(x)


class psi_2D_spinful(nn.Module):
    def __init__(
        self,
        L: int,
        hidden_layers: int,
        dim_feedforward: int,
        activation: nn.Module = nn.GELU,
        device=torch.device("cpu"),
    ):
        super().__init__()
        self.L = L
        # weights, drawn from a Gaussian distribution (for now)
        self.w_s = nn.Parameter(torch.randn(2 * (L**2), device=device))
        # instantiate MLPs
        self.f = MLP(hidden_layers, 5, dim_feedforward, 2, activation, device)
        self.g_1_prime = MLP(hidden_layers, 1, dim_feedforward, 2, activation, device)
        self.g_2_prime = MLP(hidden_layers, 1, dim_feedforward, 2, activation, device)

        self.to(device)

    # antisymmetric factor $F: \widetilde{\Lambda}^n \rightarrow \mathbb{C}$
    def F_antisymmetric(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # convert spatial coords to complex numbers e^{2\pi i x/L}
        z_x = torch.stack(
            [
                torch.cos((2 * torch.pi * x[:, :, 0]) / self.L),
                torch.sin((2 * torch.pi * x[:, :, 0]) / self.L),
                torch.cos((2 * torch.pi * x[:, :, 1]) / self.L),
                torch.sin((2 * torch.pi * x[:, :, 1]) / self.L),
                (2 * x[:, :, 2]) - 1,  # encode spin as ±1
            ],
            dim=-1,
        )  # should be (batch_size, n, 5) now
        # pass through f MLP
        f_x = torch.view_as_complex(self.f(z_x))  # complex (batch_size, n)
        vandermonde_matrix = torch.linalg.vander(
            f_x
        )  # should be of shape (batch_size, n, n)
        # take the determinant
        sign, logabsdet = torch.linalg.slogdet(vandermonde_matrix)
        F = sign * torch.exp(logabsdet)
        return F  # should be a vector of shape (batch_size)

    def eta_symmetric(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # first flatten the last dimension with an injective map
        x = (
            ((2 * self.L) * (x[:, :, 0] % self.L))
            + (2 * (x[:, :, 1] % self.L))
            + x[:, :, 2]
        )
        N_s = (
            nn.functional.one_hot(x, num_classes=2 * (self.L**2)).sum(dim=1).float()
        )  # (batch_size, 2L^2)
        # take matrix-vector product with w_s
        eta = torch.matmul(N_s, self.w_s)
        return eta  # should be a vector of shape (batch_size)

    def forward(self, x: torch.Tensor):
        # x should be integers of shape (batch_size, n, 3) with entries from \widetilde{\Lambda}
        # contains (x, y, \sigma)
        # compute the antisymmetric function F
        F = self.F_antisymmetric(x)  # shape (batch_size), dtype=complex
        # compute the symmetric function g
        # first calculate eta
        eta = self.eta_symmetric(x)  # shape (batch_size), dtype=float
        # now combine them together via:
        # \Psi = F_1 g_1 + F_2 g_2
        g_1 = torch.view_as_complex(
            self.g_1_prime(eta.unsqueeze(-1))
        )  # shape (batch_size), dtype=complex
        g_2 = torch.view_as_complex(self.g_2_prime(eta.unsqueeze(-1)))
        # the final result
        psi = (F.real * g_1) + (F.imag * g_2)  # shape (batch_size), dtype=complex
        return psi

### Test it out
- want antisymmetric

In [210]:
L = 2
B = 128
N = 2
toy_model = psi_2D_spinful(L, hidden_layers=3, dim_feedforward=256)
toy_model.eval()

psi_2D_spinful(
  (f): MLP(
    (MLP): Sequential(
      (0): Linear(in_features=5, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): GELU(approximate='none')
      (8): Linear(in_features=256, out_features=2, bias=True)
    )
  )
  (g_1_prime): MLP(
    (MLP): Sequential(
      (0): Linear(in_features=1, out_features=256, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): GELU(approximate='none')
      (4): Linear(in_features=256, out_features=256, bias=True)
      (5): GELU(approximate='none')
      (6): Linear(in_features=256, out_features=256, bias=True)
      (7): GELU(approximate='none')
      (8): Linear(in_features=256, out

In [211]:
x = torch.randint(0, L, (B, N, 3))
x[:, :, 2] = x[:, :, 2] % 2
x_swapped = x.clone()

In [215]:
# swap two particles
x_swapped[:, [0, 1], :] = x_swapped[:, [1, 0], :]

with torch.no_grad():
    y = toy_model(x_swapped)
print(y[0])

tensor(9.1877e-05-0.0001j)


### Energy Minimization Using Gradient Descent for Small Sizes
**Setup:**
Consider a $L \times L$ square lattice in momentum space. The Hubbard Hamiltonian reads:

$\renewcommand{\bsl}[1]{\boldsymbol{#1}}$
\begin{align}
    \mathcal{H}_0 &= \sum_{\bsl{k}, \sigma} \varepsilon(\bsl{k}) c_{\bsl{k}\sigma}^{\dagger}c_{\bsl{k}\sigma} \\
    \mathcal{H}_{\text{int}} &= \frac{U}{L^2} \sum_{\bsl{k}, \bsl{k'}, \bsl{q}} c_{\bsl{k}+\bsl{q}, \uparrow}^{\dagger} c_{\bsl{k}, \uparrow} c_{\bsl{k'}-\bsl{q}, \downarrow}^{\dagger} c_{\bsl{k'}, \downarrow} \\
    \mathcal{H} &= \mathcal{H}_0 + \mathcal{H}_{\text{int}},
\end{align}

where $\varepsilon(\bsl{k}) = -2t\left(\cos (k_x) + \cos(k_y)\right)$, $U$ is the on-site repulsion, and $t$ is the hopping parameter. If we convert this Hamiltonian to real space (which is what our NN expects for input), we get:

$$\mathcal{H}=-t\sum_{\langle{\bsl{i}}, \bsl{j}\rangle{},\sigma} \left(c_{\bsl{i}\sigma}^{\dagger} c_{\bsl{j}\sigma} + c_{\bsl{j}\sigma}^{\dagger} c_{\bsl{i}\sigma}\right) + U\sum_{\bsl{i}}n_{\bsl{i}\uparrow}n_{\bsl{i}\downarrow}$$

**Goal:**
We need to minimize
$$E = \frac{\langle{\Psi}|\mathcal{H}|\Psi\rangle{}}{\langle{\Psi}|\Psi\rangle{}}$$

In [200]:
# Check if CUDA is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

Using device: cpu



In [217]:
# class to implement the Hubbard Hamiltonian in real space
class HamiltonianOperator:
    def __init__(self, L: int, n: int, diff: int, t: int, U: int):
        self.L = L
        self.n = n
        self.diff = diff
        self.t = t
        self.U = U

        # build H
        self.build_H()

    def generate_basis(self):
        # diff = n_spin_up - n_spin_down
        # diff and n should have the same parity
        n_spin_up = (self.n + self.diff) // 2
        n_spin_down = (self.n - self.diff) // 2

        positions_up = []
        positions_down = []
        for i, j in itertools.product(range(self.L), repeat=2):
            positions_up.append((i, j, 1))
            positions_down.append((i, j, 0))

        self.basis = []
        for up, down in itertools.product(
            itertools.combinations(positions_up, n_spin_up),
            itertools.combinations(positions_down, n_spin_down),
        ):
            state = [*up, *down]

            # sort it
            self.basis.append(
                tuple(
                    sorted(
                        state, key=lambda x: ((2 * self.L) * x[0]) + (2 * x[1]) + x[2]
                    )
                )
            )
        # lookup table
        self.state_to_index = {state: idx for idx, state in enumerate(self.basis)}

    # function to implement the diagonal interaction term
    def U_interaction(self, state: tuple):
        # state should be tuple of tuples containing (x, y, \sigma)
        # loop over all lattice sites
        count = 0
        for i_x, i_y in itertools.product(range(self.L), repeat=2):
            if sum((pos[0] == i_x and pos[1] == i_y) for pos in state) == 2:
                count += 1
        return self.U * count

    # helper "creation"/annihilation operators
    def c_op(self, pos: tuple, state: tuple):
        index = state.index(pos)
        return index, state[:index] + state[index + 1 :]

    def c_op_dagger(self, pos: tuple, state: tuple):
        state = tuple(
            sorted(
                state + (pos,), key=lambda x: ((2 * self.L) * x[0]) + (2 * x[1]) + x[2]
            )
        )
        return state.index(pos), state

    # the function to build the matrix representation of the Hamiltonian in real space
    def build_H(self):
        self.generate_basis()
        self.H = np.zeros((len(self.basis), len(self.basis)), dtype=np.complex128)
        # loop over all the basis states
        for idx, state in enumerate((self.basis)):
            # first compute the diagonal entries
            self.H[idx, idx] += self.U_interaction(state)
            # now for the off-diagonal entries
            for i_x, i_y, sigma in itertools.product(
                range(self.L), range(self.L), (0, 1)
            ):
                # calculate adjacent lattice site
                directions = [(1, 0), (-1, 0), (0, 1), (0, -1)]
                for dx, dy in directions:
                    j_x = (i_x + dx) % self.L
                    j_y = (i_y + dy) % self.L
                    i_pos = (i_x, i_y, sigma)
                    j_pos = (j_x, j_y, sigma)
                    # to track the phase
                    exponent = 0
                    # first term
                    if j_pos not in state:
                        continue
                    # annihilate j
                    temp, new_state = self.c_op(j_pos, state)
                    exponent += temp
                    if i_pos in new_state:
                        continue
                    # create i
                    temp, new_state = self.c_op_dagger(i_pos, new_state)
                    exponent += temp
                    # calculate phase
                    if exponent % 2 == 0:
                        phase = 1
                    else:
                        phase = -1
                    # get index
                    self.H[idx, self.state_to_index[new_state]] += phase * (-self.t)

In [221]:
# test if it's correct via ED
toy_hamiltonian = HamiltonianOperator(2, 2, 0, t=1, U=2)
eigvals, eigvecs = np.linalg.eigh(toy_hamiltonian.H)
print(eigvals[0])  # matches with my C++ ED!

-7.5705217296593155
